# A9-CWO · Request 07 — Private Drive inventory
**Authority:** GitHub `main` versions the implementation. **Source originals:** authenticated Google Drive. **Outputs:** private user-owned Drive storage, **never GitHub**.

Run all cells in an **authorized Google Colab account**. You must approve Google's authentication and Drive mount dialogs. This is a metadata-only collector; it does not download drawing bytes, move source files, publish candidate details, or approve construction revisions.

This run inventories the **currently registered nine roots only**. Six P001/P002 source pointers remain outside those registered roots and **must not** be force-mapped or marked complete. Independently authorized alternate roots require a separate private reviewed-root map and a new run.

**Safety:** no public resource admission; duplicate Drive IDs are reconciled, conflicting cross-project associations fail closed; outputs never overwrite previous runs.

In [ ]:
%pip -q install google-api-python-client google-auth
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")
print("Authenticated Google API + private Drive mount ready.")

In [ ]:
from pathlib import Path
import datetime, hashlib, json, subprocess, sys

REPOSITORY = "https://github.com/FabinGurung/JP_Communication-Command-Centre-and-Resource-Hub.git"
WORKTREE = Path("/content/a9_cwo_req07_versioned_source")
if WORKTREE.exists():
    raise RuntimeError("Working source directory already exists. Start a fresh Colab runtime to avoid ambiguous code versions.")
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(WORKTREE)], check=True)
SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(WORKTREE), "rev-parse", "HEAD"], text=True).strip()
COLLECTOR = WORKTREE / "scripts/collect_private_drive_resource_metadata.py"
PREVIEW = WORKTREE / "scripts/preview_resource_intake.py"
assert COLLECTOR.exists() and PREVIEW.exists(), "Versioned governed scripts missing"
print("GitHub source commit:", SOURCE_COMMIT)

In [ ]:
PRIVATE_BASE = Path("/content/drive/MyDrive/JP_AEC_PRIVATE_A9_CWO/REQ07_RESOURCE_INVENTORY")
PRIVATE_BASE.mkdir(parents=True, exist_ok=True)
RUN_ID = "REQ07_" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = PRIVATE_BASE / RUN_ID
RUN_DIR.mkdir(exist_ok=False)
INVENTORY = RUN_DIR / "registered_roots_metadata.jsonl"
RECEIPT = RUN_DIR / "registered_roots_metadata.receipt.json"
CANDIDATES = RUN_DIR / "private_candidate_review.json"
assert WORKTREE not in RUN_DIR.parents and RUN_DIR != WORKTREE
print("PRIVATE OUTPUT:", RUN_DIR)
print("Never commit this directory, its filenames, or its file IDs to public GitHub.")

In [ ]:
subprocess.run([
    sys.executable, str(COLLECTOR),
    "--registered-roots",
    "--private-jsonl", str(INVENTORY),
    "--receipt", str(RECEIPT)
], cwd=str(WORKTREE), check=True)
receipt = json.loads(RECEIPT.read_text(encoding="utf-8"))
raw = INVENTORY.read_bytes()
assert hashlib.sha256(raw).hexdigest() == receipt["jsonl_sha256"], "SHA-256 mismatch"
assert len(raw.splitlines()) == receipt["files_and_folders"], "Record count mismatch"
assert receipt["public_admissions"] == 0 and receipt["drive_modifications"] == 0 and receipt["byte_downloads"] == 0
print("PRIVATE RECEIPT VERIFIED")
print("Projects:", receipt["indexed_project_count"], "Roots:", receipt["project_root_count"], "Folders:", receipt["visited_folders"], "Unique entries:", receipt["files_and_folders"])
print("Known source coverage (still not project-owner authorization):", receipt["registered_source_coverage"])

In [ ]:
subprocess.run([
    sys.executable, str(PREVIEW),
    "--input", str(INVENTORY),
    "--receipt", str(RECEIPT),
    "--output", str(CANDIDATES),
], cwd=str(WORKTREE), check=True)
review = json.loads(CANDIDATES.read_text(encoding="utf-8"))
assert review["automatic_publication"] is False
assert len(review["rows"]) <= receipt["files_and_folders"]
summary = {
    "type": "A9_CWO_REQ07_PRIVATE_RUN_SUMMARY",
    "source_git_commit": SOURCE_COMMIT,
    "run_id": RUN_ID,
    "provider": "Google Drive files.list page-token authenticated metadata read",
    "inventory_sha256": receipt["jsonl_sha256"],
    "receipt_files_and_folders": receipt["files_and_folders"],
    "receipt_root_count": receipt["project_root_count"],
    "receipt_visited_folders": receipt["visited_folders"],
    "registered_source_coverage": receipt["registered_source_coverage"],
    "review_status_counts": {state:sum(x["review_status"] == state for x in review["rows"])
        for state in ("ALREADY_INDEXED", "QUARANTINED", "REVIEW_REQUIRED")},
    "public_resource_admissions": 0,
    "source_original_changes": 0,
    "owner_authorized_alternate_roots": False,
    "request07_complete": False,
}
summary_path = RUN_DIR / "private_run_summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
assert json.loads(summary_path.read_text(encoding="utf-8")) == summary
print("PRIVATE OUTPUT READBACK PASS:", RUN_DIR)
print("REVIEW COUNTS:", summary["review_status_counts"])
print("NOT PUBLICATION-READY: P001/P002 alternate-source ownership and per-file revision/privacy decisions are still pending.")

## Subsequent owner review (separate authorization)
The private `registered_roots_metadata.jsonl` and `private_candidate_review.json` must be inspected in controlled storage. Owner-verified alternate folders, if any, should be supplied through a **private** JSON mapping (`--roots /private/reviewed_roots.json`) in a **new run directory**. Never infer ownership solely from the presence of a parent folder. Do not move any original sources. Only approved, source-verified, non-private, current-revision files may be considered for a separately gated website catalog publication.